# Model Optimization: Quantization

In this notebook, we'll apply quantization techniques to our models using distributed processing. Instead of running the quantization on our notebook instance, we'll launch separate SageMaker Processing jobs to perform the quantization on more powerful instances.

## What is Quantization?

Quantization is a technique that reduces the precision of the numbers used to represent a model's parameters. For example, converting 32-bit floating point numbers to 8-bit integers. This significantly reduces model size and can improve inference speed, often with minimal impact on accuracy.

### Benefits of Quantization:
- **Reduced Model Size**: Smaller models require less storage and memory
- **Faster Inference**: Lower precision calculations can be faster, especially on hardware with specialized support
- **Lower Memory Bandwidth**: Smaller models require less memory bandwidth, which can be a bottleneck
- **Energy Efficiency**: Lower precision calculations consume less power

### Types of Quantization We'll Explore:
- **Dynamic Quantization**: Weights are quantized ahead of time, but activations are quantized dynamically during inference
- **Static Quantization**: Both weights and activations are quantized ahead of time
- **Quantization-Aware Training**: The model is trained with simulated quantization to improve accuracy

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform quantization on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
import os
import json
import time
import pandas as pd
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
import time
from IPython.display import clear_output

# Import our utility functions
from optimization_utils import analyze_job_failure, handle_processing_error, save_checkpoint

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE
%store -r baseline_metrics
%store -r endpoint_names

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    SAGEMAKER_ROLE_ARN = "YOUR_ROLE_ARN_HERE"  # Update this value
    OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Default optimization instance type
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN
    %store OPTIMIZATION_INSTANCE_TYPE

## 3. Load Baseline Metrics

Load the baseline metrics from the previous notebook for comparison.

In [ ]:
# Check if baseline metrics were loaded
if 'baseline_metrics' in locals() and baseline_metrics:
    print("Baseline metrics loaded successfully.")
    print(f"Found metrics for {len(baseline_metrics)} models.")
else:
    print("⚠️ Baseline metrics not found.")
    print("Please run the previous notebook (02_baseline_evaluation.ipynb) first.")
    
    # Try to load from file if available
    try:
        with open('baseline_metrics.json', 'r') as f:
            baseline_metrics = json.load(f)
        print(f"Loaded baseline metrics from file for {len(baseline_metrics)} models.")
    except FileNotFoundError:
        print("Could not find baseline_metrics.json file.")
        baseline_metrics = {}

# Extract model information from baseline metrics
model_info = {}
for model_key, metrics in baseline_metrics.items():
    model_info[model_key] = {
        "model_name": metrics["model_name"],
        "task": metrics["task"]
    }

print("\nModels to quantize:")
for model_key, info in model_info.items():
    print(f"- {model_key}: {info['model_name']} ({info['task']})")

## 4. Examine Quantization Script

Let's examine the quantization script that will be executed on the SageMaker Processing instances.

In [ ]:
# Display the quantization script with syntax highlighting
%pycat quantization_script.py

## 5. Make Script Executable and Upload to S3

Now we'll make the script executable and upload it to S3 so it can be accessed by the SageMaker Processing jobs.

In [ ]:
# Make the script executable
!chmod +x quantization_script.py

# Upload the quantization script to S3
s3_client = boto3.client('s3')
s3_client.upload_file(
    'quantization_script.py', 
    S3_BUCKET, 
    'scripts/quantization_script.py'
)

print(f"Uploaded quantization script to s3://{S3_BUCKET}/scripts/quantization_script.py")

## 6. Launch Distributed Quantization Jobs

Now we'll set up and launch the SageMaker Processing jobs to perform quantization. Each model will be processed in a separate job, allowing for parallel processing.

### SageMaker Processing Components:
- **PyTorchProcessor**: A specialized processor for PyTorch workloads
- **Instance Type**: We'll use the instance type specified in the workshop settings
- **Role**: The SageMaker execution role with necessary permissions
- **Base Job Name**: A prefix for the SageMaker job names

The processor will run our quantization script on the specified instance type, with access to the model information stored in S3.

In [ ]:
# Define the instance type to use for quantization
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="1.13.1",
    py_version="py39",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="model-quantization",
    sagemaker_session=sagemaker_session,
    command=["python3"]  # Explicitly specify the command to run the script
)

In [ ]:
# Launch quantization jobs for each model
quantization_jobs = {}

for model_key in model_info.keys():
    print(f"\nLaunching quantization job for {model_key}...")
    
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/scripts/quantization_script.py',
            destination='/opt/ml/processing/input/code/quantization_script.py'
        ),
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data/model_info.json'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=f's3://{S3_BUCKET}/optimization/outputs/{model_key}'
        )
    ]
    
    # Run the processing job
    try:
        job = processor.run(
            source_dir=None,  # Don't use source_dir
            code='/opt/ml/processing/input/code/quantization_script.py',  # Use absolute path
            inputs=inputs,
            outputs=outputs,
            arguments=[
                '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
                '--output-dir', '/opt/ml/processing/output',
                '--quantization-method', 'dynamic',
                '--quantization-bits', '8'
            ],
            logs=True  # Enable CloudWatch Logs streaming
        )
        quantization_jobs[model_key] = job
        print(f"Launched job: {job.job_name}")
    except Exception as e:
        print(f"Error launching job for {model_key}: {e}")
        handle_processing_error(e, f"quantization-{model_key}")

print(f"\nLaunched {len(quantization_jobs)} quantization jobs")

## 7. Monitor Job Status

Now we'll monitor the status of the quantization jobs. We'll periodically check the status of each job and display it.

In [ ]:
# Monitor job status
sagemaker_client = boto3.client('sagemaker')

# Initialize status tracking
job_status = {model_key: "InProgress" for model_key in quantization_jobs.keys()}
start_time = time.time()

# Monitor until all jobs are complete
try:
    while "InProgress" in job_status.values() or "Starting" in job_status.values():
        # Clear previous output
        clear_output(wait=True)
        
        # Calculate elapsed time
        elapsed_time = time.time() - start_time
        elapsed_minutes = int(elapsed_time // 60)
        elapsed_seconds = int(elapsed_time % 60)
        
        print(f"Monitoring quantization jobs... (Elapsed time: {elapsed_minutes:02d}:{elapsed_seconds:02d})\n")
        
        # Check status of each job
        for model_key, job in quantization_jobs.items():
            try:
                response = sagemaker_client.describe_processing_job(
                    ProcessingJobName=job.job_name
                )
                status = response["ProcessingJobStatus"]
                job_status[model_key] = status
                
                # Print status with color coding
                status_display = status
                if status == "Completed":
                    status_display = "✅ Completed"
                elif status == "Failed":
                    status_display = "❌ Failed"
                elif status == "Stopped":
                    status_display = "⚠️ Stopped"
                elif status == "InProgress":
                    status_display = "⏳ InProgress"
                
                print(f"{model_key}: {status_display}")
                
            except Exception as e:
                print(f"{model_key}: Error checking status - {e}")
        
        # If all jobs are complete, break the loop
        if "InProgress" not in job_status.values() and "Starting" not in job_status.values():
            break
        
        # Wait before checking again
        time.sleep(30)
    
    # Final status report
    print("\nAll jobs completed!")
    print("\nFinal status:")
    for model_key, status in job_status.items():
        if status == "Failed":
            print(f"{model_key}: ❌ {status}")
            analyze_job_failure(quantization_jobs[model_key].job_name)
        else:
            print(f"{model_key}: {'✅' if status == 'Completed' else '⚠️'} {status}")
    
except Exception as e:
    print(f"Error monitoring jobs: {e}")

## 8. Collect Quantization Results

Now we'll collect the results of the quantization jobs and analyze them.

In [ ]:
# Collect quantization metrics
quantization_metrics = {}

for model_key in model_info.keys():
    if job_status.get(model_key) == "Completed":
        try:
            # Download metrics file from S3
            s3_client.download_file(
                S3_BUCKET,
                f'optimization/outputs/{model_key}/quantization_metrics.json',
                f'{model_key}_quantization_metrics.json'
            )
            
            # Load metrics
            with open(f'{model_key}_quantization_metrics.json', 'r') as f:
                metrics = json.load(f)
            
            # Add to collected metrics
            quantization_metrics.update(metrics)
            print(f"Collected metrics for {model_key}")
            
        except Exception as e:
            print(f"Error collecting metrics for {model_key}: {e}")
    else:
        print(f"Skipping {model_key} as job did not complete successfully")

# Save all metrics to a single file
with open('quantization_metrics.json', 'w') as f:
    json.dump(quantization_metrics, f, indent=2)

# Store for later use
%store quantization_metrics

print(f"\nCollected quantization metrics for {len(quantization_metrics)} models")

## 9. Analyze Quantization Results

Now let's analyze the results of the quantization to understand the improvements achieved.

In [ ]:
# Create a DataFrame with the metrics for better display
if quantization_metrics:
    model_names = []
    baseline_sizes = []
    quantized_sizes = []
    size_reductions = []
    baseline_times = []
    quantized_times = []
    time_improvements = []
    
    for model_key, metrics in quantization_metrics.items():
        model_names.append(metrics["model_name"])
        baseline_sizes.append(metrics["baseline_size_mb"])
        quantized_sizes.append(metrics["quantized_size_mb"])
        size_reductions.append(metrics["size_reduction_percent"])
        baseline_times.append(metrics["baseline_inference_time_ms"])
        quantized_times.append(metrics["quantized_inference_time_ms"])
        time_improvements.append(metrics["time_improvement_percent"])
    
    # Create a DataFrame
    metrics_df = pd.DataFrame({
        "Model": model_names,
        "Baseline Size (MB)": baseline_sizes,
        "Quantized Size (MB)": quantized_sizes,
        "Size Reduction (%)": size_reductions,
        "Baseline Time (ms)": baseline_times,
        "Quantized Time (ms)": quantized_times,
        "Time Improvement (%)": time_improvements
    })
    
    # Display the metrics table
    metrics_df
else:
    print("No quantization metrics available for analysis.")

## 10. Visualize Quantization Results

Let's create some visualizations to better understand the improvements achieved through quantization.

In [ ]:
# Create visualizations
if quantization_metrics:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    plt.figure(figsize=(15, 10))
    
    # Model size comparison
    plt.subplot(2, 2, 1)
    x = range(len(model_names))
    width = 0.35
    plt.bar(x, baseline_sizes, width, label='Baseline')
    plt.bar([i + width for i in x], quantized_sizes, width, label='Quantized')
    plt.xlabel('Model')
    plt.ylabel('Size (MB)')
    plt.title('Model Size Comparison')
    plt.xticks([i + width/2 for i in x], model_names, rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()
    
    # Inference time comparison
    plt.subplot(2, 2, 2)
    plt.bar(x, baseline_times, width, label='Baseline')
    plt.bar([i + width for i in x], quantized_times, width, label='Quantized')
    plt.xlabel('Model')
    plt.ylabel('Inference Time (ms)')
    plt.title('Inference Time Comparison')
    plt.xticks([i + width/2 for i in x], model_names, rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()
    
    # Size reduction percentage
    plt.subplot(2, 2, 3)
    plt.bar(model_names, size_reductions)
    plt.xlabel('Model')
    plt.ylabel('Size Reduction (%)')
    plt.title('Size Reduction from Quantization')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    # Time improvement percentage
    plt.subplot(2, 2, 4)
    plt.bar(model_names, time_improvements)
    plt.xlabel('Model')
    plt.ylabel('Time Improvement (%)')
    plt.title('Inference Time Improvement from Quantization')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    plt.show()
else:
    print("No quantization metrics available for visualization.")

## 11. Next Steps

Now that we've applied quantization to our models, we'll explore pruning techniques in the next notebook to further reduce model size and improve inference speed.

### What We've Learned:
- How to apply quantization to transformer models
- How to measure the impact of quantization on model size and inference time
- How to use SageMaker Processing jobs for distributed model optimization

### What's Next - Pruning:
Pruning is a technique that removes unnecessary connections in a neural network. This can significantly reduce model size and improve inference speed, often with minimal impact on accuracy.